In [12]:
from pyiceberg.catalog import load_catalog
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import sedona.db
import pyarrow as pa

In [6]:
sd = sedona.db.connect()

In [2]:
# Create a temporary location for Iceberg
!mkdir /tmp/warehouse

In [3]:
# set up the catalog
warehouse_path = "/tmp/warehouse"
catalog = load_catalog(
    "default",
    **{
        'type': 'sql',
        "uri": f"sqlite:///{warehouse_path}/pyiceberg_catalog.db",
        "warehouse": f"file://{warehouse_path}",
    },
)

In [4]:
# df = pq.read_table("/Users/matthewpowers/data/tpch_sf1/nation.parquet")

In [7]:
countries = sd.read_parquet(
    "https://raw.githubusercontent.com/geoarrow/geoarrow-data/v0.2.0/natural-earth/files/natural-earth_countries_geo.parquet"
)

In [35]:
countries.to_view("countries", True)
df = sd.sql(
    "select ARROW_CAST(name, 'Utf8') as name, ARROW_CAST(continent, 'Utf8') as continent, ST_AsText(geometry) as geometry_wkt from countries"
).limit(10)
# table_path = "/tmp/delta_with_wkt"
# write_deltalake(table_path, df.to_pandas(), mode="overwrite")

In [36]:
df.schema

SedonaSchema with 3 fields:
  name: utf8<Utf8>
  continent: utf8<Utf8>
  geometry_wkt: utf8<Utf8>

In [37]:
df.to_arrow_table().schema

name: string
continent: string
geometry_wkt: string

In [29]:
catalog.list_namespaces()

[('default',)]

In [38]:
# create a new iceberg table
from pyiceberg.exceptions import NamespaceAlreadyExistsError
try:
    catalog.create_namespace("default")
except NamespaceAlreadyExistsError:
    pass

if catalog.table_exists("default.countries"):
    catalog.drop_table("default.countries")

table = catalog.create_table(
    "default.countries",
    schema=df.to_arrow_table().schema,
)

In [39]:
# Append the dataframe to the table
table.append(df.to_arrow_table())

In [40]:
# check length of table
len(table.scan().to_arrow())

10

## Read Iceberg table into SedonaDB via Arrow

In [41]:
# Load the table (namespace.table_name)
table = catalog.load_table("default.countries")

# Scan and load as a PyArrow table
arrow_table = table.scan().to_arrow()

# Convert to SedonaDB DataFrame
df = sd.create_data_frame(arrow_table)

In [42]:
df.show()

┌─────────────────────────────┬───────────────┬────────────────────────────────────────────────────┐
│             name            ┆   continent   ┆                    geometry_wkt                    │
│             utf8            ┆      utf8     ┆                        utf8                        │
╞═════════════════════════════╪═══════════════╪════════════════════════════════════════════════════╡
│ Fiji                        ┆ Oceania       ┆ MULTIPOLYGON(((180 -16.067132663642447,180 -16.55… │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ United Republic of Tanzania ┆ Africa        ┆ POLYGON((33.90371119710453 -0.9500000000000001,34… │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Western Sahara              ┆ Africa        ┆ POLYGON((-8.665589565454809 27.656425889592356,-8… │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌

In [43]:
df.to_view("my_table", True)
res = sd.sql("""
SELECT
  name,
  continent,
  ST_GeomFromWKT(geometry_wkt) as geom
from my_table
""")
res.show()

┌─────────────────────────────┬───────────────┬────────────────────────────────────────────────────┐
│             name            ┆   continent   ┆                        geom                        │
│             utf8            ┆      utf8     ┆                      geometry                      │
╞═════════════════════════════╪═══════════════╪════════════════════════════════════════════════════╡
│ Fiji                        ┆ Oceania       ┆ MULTIPOLYGON(((180 -16.067132663642447,180 -16.55… │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ United Republic of Tanzania ┆ Africa        ┆ POLYGON((33.90371119710453 -0.9500000000000001,34… │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ Western Sahara              ┆ Africa        ┆ POLYGON((-8.665589565454809 27.656425889592356,-8… │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌